#**스마트폰 센서 데이터 기반 모션 분류**
# 단계2 : 기본 모델링


## 0.미션

* 데이터 전처리
    * 가변수화, 데이터 분할, NaN 확인 및 조치, 스케일링 등 필요한 전처리 수행
* 다양한 알고리즘으로 분류 모델 생성
    * 최소 4개 이상의 알고리즘을 적용하여 모델링 수행
    * 성능 비교
    * 각 모델의 성능을 저장하는 별도 데이터 프레임을 만들고 비교
* 옵션 : 다음 사항은 선택사항입니다. 시간이 허용하는 범위 내에서 수행하세요.
    * 상위 N개 변수를 선정하여 모델링 및 성능 비교
        * 모델링에 항상 모든 변수가 필요한 것은 아닙니다.
        * 변수 중요도 상위 N개를 선정하여 모델링하고 타 모델과 성능을 비교하세요.
        * 상위 N개를 선택하는 방법은, 변수를 하나씩 늘려가며 모델링 및 성능 검증을 수행하여 적절한 지점을 찾는 것입니다.

## 1.환경설정

In [210]:
import random
import numpy as np
import os

# 1. 파이썬 자체 난수 고정
random.seed(42)

# 2. 넘파이 난수 고정
np.random.seed(42)

# 3. 텐서플로우 사용하는 경우 (딥러닝)
import tensorflow as tf
tf.random.set_seed(42)

# 4. (선택) 실행 환경 내에서 다른 난수도 통제하고 싶을 때
os.environ['PYTHONHASHSEED'] = '42'

In [211]:
# # 한글 깨짐 방지 위해 구글 코랩에 나눔폰트 설치.
# import sys
# if 'google.colab' in sys.modules:
#     !echo 'debconf debconf/frontend select Noninteractive' | debconf-set-selections
#     !sudo apt-get -qq -y install fonts-nanum
#     import matplotlib.font_manager as fm
#     font_files = fm.findSystemFonts(fontpaths=['/usr/share/fonts/truetype/nanum'])
#     for fpath in font_files:
#         fm.fontManager.addfont(fpath)

In [212]:
# # 나눔고딕으로 폰트 변경
# import matplotlib.pyplot as plt
# plt.rcParams['font.family'] = 'NanumGothic'

In [213]:
# # matplot의 폰트와 폰트 크기 확인
# print(plt.rcParams['font.family'], plt.rcParams['font.size'])

### (1) 라이브러리 불러오기

* 세부 요구사항
    - 기본적으로 필요한 라이브러리를 import 하도록 코드가 작성되어 있습니다.
    - 필요하다고 판단되는 라이브러리를 추가하세요.

In [214]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 필요하다고 판단되는 라이브러리를 추가하세요.
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
# from xgboost import XGBClassifier
# import lightgbm as lgb
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import *
import joblib

* 함수 생성

In [215]:
# 변수의 특성 중요도 계산하기
def plot_feature_importance(importance, names, result_only = False, topn = 'all'):
    feature_importance = np.array(importance)
    feature_name = np.array(names)

    data={'feature_name':feature_name,'feature_importance':feature_importance}
    fi_temp = pd.DataFrame(data)

    #변수의 특성 중요도 순으로 정렬하기
    fi_temp.sort_values(by=['feature_importance'], ascending=False,inplace=True)
    fi_temp.reset_index(drop=True, inplace = True)

    if topn == 'all' :
        fi_df = fi_temp.copy()
    else :
        fi_df = fi_temp.iloc[:topn]

    #변수의 특성 중요도 그래프로 그리기
    if result_only == False :
        plt.figure(figsize=(10,20))
        sns.barplot(x='feature_importance', y='feature_name', data = fi_df)

        plt.xlabel('importance')
        plt.ylabel('feature name')
        plt.grid()

    return fi_df

### (2) 데이터 불러오기

* 주어진 데이터셋
    * data01_train.csv : 학습 및 검증용
* 세부 요구사항
    - 전체 데이터 'data01_train.csv' 를 불러와 'data' 이름으로 저장합니다.
        - data에서 변수 subject는 삭제합니다.
    - 데이터프레임에 대한 기본 정보를 확인합니다.( .head(), .shape 등)

#### 1) 데이터 로딩

In [216]:
train_data = pd.read_csv('data01_train.csv')

In [217]:
test_data = pd.read_csv('data01_test.csv')

#### 2) 기본 정보 조회

In [218]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5881 entries, 0 to 5880
Columns: 563 entries, tBodyAcc-mean()-X to Activity
dtypes: float64(561), int64(1), object(1)
memory usage: 25.3+ MB


In [219]:
train_data.head()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,Activity
0,0.288508,-0.009196,-0.103362,-0.988986,-0.962797,-0.967422,-0.989000,-0.962596,-0.965650,-0.929747,...,-0.816696,-0.042494,-0.044218,0.307873,0.072790,-0.601120,0.331298,0.165163,21,STANDING
1,0.265757,-0.016576,-0.098163,-0.989551,-0.994636,-0.987435,-0.990189,-0.993870,-0.987558,-0.937337,...,-0.693515,-0.062899,0.388459,-0.765014,0.771524,0.345205,-0.769186,-0.147944,15,LAYING
2,0.278709,-0.014511,-0.108717,-0.997720,-0.981088,-0.994008,-0.997934,-0.982187,-0.995017,-0.942584,...,-0.829311,0.000265,-0.525022,-0.891875,0.021528,-0.833564,0.202434,-0.032755,11,STANDING
3,0.289795,-0.035536,-0.150354,-0.231727,-0.006412,-0.338117,-0.273557,0.014245,-0.347916,0.008288,...,-0.408956,-0.255125,0.612804,0.747381,-0.072944,-0.695819,0.287154,0.111388,17,WALKING
4,0.394807,0.034098,0.091229,0.088489,-0.106636,-0.388502,-0.010469,-0.109680,-0.346372,0.584131,...,-0.563437,-0.044344,-0.845268,-0.974650,-0.887846,-0.705029,0.264952,0.137758,17,WALKING_DOWNSTAIRS


In [220]:
train_data.describe()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject
count,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,...,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000
mean,0.274811,-0.017799,-0.109396,-0.603138,-0.509815,-0.604058,-0.628151,-0.525944,-0.605374,-0.465490,...,-0.305883,-0.623548,0.008524,-0.001185,0.009340,-0.007099,-0.491501,0.059299,-0.054594,17.381568
std,0.067614,0.039422,0.058373,0.448807,0.501815,0.417319,0.424345,0.485115,0.413043,0.544995,...,0.322808,0.310371,0.339730,0.447197,0.608190,0.476738,0.509069,0.297340,0.278479,8.938316
min,-0.503823,-0.684893,-1.000000,-1.000000,-0.999844,-0.999667,-1.000000,-0.999419,-1.000000,-1.000000,...,-0.979261,-0.999765,-0.976580,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-0.980143,1.000000
25%,0.262919,-0.024877,-0.121051,-0.992774,-0.977680,-0.980127,-0.993602,-0.977865,-0.980112,-0.936067,...,-0.541969,-0.845985,-0.122361,-0.294369,-0.481718,-0.373345,-0.811397,-0.018203,-0.141555,8.000000
50%,0.277154,-0.017221,-0.108781,-0.943933,-0.844575,-0.856352,-0.948501,-0.849266,-0.849896,-0.878729,...,-0.342923,-0.712677,0.010278,0.005146,0.011448,-0.000847,-0.709441,0.182893,0.003951,19.000000
75%,0.288526,-0.010920,-0.098163,-0.242130,-0.034499,-0.262690,-0.291138,-0.068857,-0.268539,-0.013690,...,-0.127371,-0.501158,0.154985,0.285030,0.499857,0.356236,-0.511330,0.248435,0.111932,26.000000
max,1.000000,1.000000,1.000000,1.000000,0.916238,1.000000,1.000000,0.967664,1.000000,1.000000,...,0.989538,0.956845,1.000000,1.000000,0.998702,0.996078,0.977344,0.478157,1.000000,30.000000


## **2. 데이터 전처리**

* 가변수화, 데이터 분할, NaN 확인 및 조치, 스케일링 등 필요한 전처리를 수행한다.


In [221]:
train_data.drop('subject', axis=1, inplace=True)

In [222]:
train_data.head(5)

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",Activity
0,0.288508,-0.009196,-0.103362,-0.988986,-0.962797,-0.967422,-0.989000,-0.962596,-0.965650,-0.929747,...,-0.487737,-0.816696,-0.042494,-0.044218,0.307873,0.072790,-0.601120,0.331298,0.165163,STANDING
1,0.265757,-0.016576,-0.098163,-0.989551,-0.994636,-0.987435,-0.990189,-0.993870,-0.987558,-0.937337,...,-0.237820,-0.693515,-0.062899,0.388459,-0.765014,0.771524,0.345205,-0.769186,-0.147944,LAYING
2,0.278709,-0.014511,-0.108717,-0.997720,-0.981088,-0.994008,-0.997934,-0.982187,-0.995017,-0.942584,...,-0.535287,-0.829311,0.000265,-0.525022,-0.891875,0.021528,-0.833564,0.202434,-0.032755,STANDING
3,0.289795,-0.035536,-0.150354,-0.231727,-0.006412,-0.338117,-0.273557,0.014245,-0.347916,0.008288,...,-0.004012,-0.408956,-0.255125,0.612804,0.747381,-0.072944,-0.695819,0.287154,0.111388,WALKING
4,0.394807,0.034098,0.091229,0.088489,-0.106636,-0.388502,-0.010469,-0.109680,-0.346372,0.584131,...,-0.157832,-0.563437,-0.044344,-0.845268,-0.974650,-0.887846,-0.705029,0.264952,0.137758,WALKING_DOWNSTAIRS


In [223]:
sum(train_data.isna().sum())

0

### (1) 데이터 분할1 : x, y

* 세부 요구사항
    - x, y로 분할합니다.

In [224]:
static_activity = (train_data['Activity']=='LAYING') | (train_data['Activity']=='SITTING') | (train_data['Activity']=='STANDING')
train_data = train_data[static_activity]

In [225]:
static_activity = (test_data['Activity']=='LAYING') | (test_data['Activity']=='SITTING') | (test_data['Activity']=='STANDING')
test_data = test_data[static_activity]

In [226]:
train_data.head(5)

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",Activity
0,0.288508,-0.009196,-0.103362,-0.988986,-0.962797,-0.967422,-0.989000,-0.962596,-0.965650,-0.929747,...,-0.487737,-0.816696,-0.042494,-0.044218,0.307873,0.072790,-0.601120,0.331298,0.165163,STANDING
1,0.265757,-0.016576,-0.098163,-0.989551,-0.994636,-0.987435,-0.990189,-0.993870,-0.987558,-0.937337,...,-0.237820,-0.693515,-0.062899,0.388459,-0.765014,0.771524,0.345205,-0.769186,-0.147944,LAYING
2,0.278709,-0.014511,-0.108717,-0.997720,-0.981088,-0.994008,-0.997934,-0.982187,-0.995017,-0.942584,...,-0.535287,-0.829311,0.000265,-0.525022,-0.891875,0.021528,-0.833564,0.202434,-0.032755,STANDING
7,0.272026,-0.001329,-0.125491,-0.992068,-0.912985,-0.972451,-0.994752,-0.943141,-0.976428,-0.925446,...,-0.258503,-0.704995,-0.024442,0.076332,0.741277,0.729812,-0.817201,0.037746,0.136129,STANDING
8,0.284338,0.021956,-0.006925,-0.980153,-0.838394,-0.782357,-0.983683,-0.816199,-0.743923,-0.914011,...,-0.052863,-0.400197,0.021212,-0.009465,-0.282762,0.563343,-0.782072,0.242834,-0.025285,STANDING


In [227]:
X_train = train_data.drop('Activity', axis=1)
y_train = train_data.loc[:, 'Activity']

In [228]:
X_test = test_data.drop(['subject', 'Activity'], axis=1)
y_test = test_data.loc[:, 'Activity']

In [229]:
y_train.unique()

array(['STANDING', 'LAYING', 'SITTING'], dtype=object)

In [230]:
y_test.unique()

array(['SITTING', 'STANDING', 'LAYING'], dtype=object)

### (2) 스케일링(필요시)


* 세부 요구사항
    - 스케일링을 필요로 하는 알고리즘 사용을 위해서 코드 수행
    - min-max 방식 혹은 standard 방식 중 한가지 사용.

In [231]:
# 스케일링 필요 없다고 판단

### (3) 데이터분할2 : train, validation

* 세부 요구사항
    - train : val = 8 : 2 혹은 7 : 3
    - random_state 옵션을 사용하여 다른 모델과 비교를 위해 성능이 재현되도록 합니다.

In [232]:
# X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

## **3. 기본 모델링**



* 세부 요구사항
    - 최소 4개 이상의 알고리즘을 적용하여 모델링을 수행한다.
    - 각 알고리즘별로 전체 변수로 모델링, 상위 N개 변수를 선택하여 모델링을 수행하고 성능 비교를 한다.
    - (옵션) 알고리즘 중 1~2개에 대해서, 변수 중요도 상위 N개를 선정하여 모델링하고 타 모델과 성능을 비교.
        * 상위 N개를 선택하는 방법은, 변수를 하나씩 늘려가며 모델링 및 성능 검증을 수행하여 적절한 지점을 찾는 것이다.

In [233]:
acc_df = pd.DataFrame({'모델': [],
                       'default_정확도(교차검증)': [],
                       'default_정확도(테스트)': [],
                       'default_정확도(fi_교차검증)': [],
                       'default_정확도(fi_테스트)': []
                      })
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트)


In [234]:
kappa_df = pd.DataFrame({'모델': [],
                       'default_kappa(교차검증)': [],
                       'default_kappa(테스트)': [],
                      })
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트)


In [235]:
kappa_scorer = make_scorer(cohen_kappa_score)

### (1) 알고리즘1 : RandomForest

In [236]:
rf_model = RandomForestClassifier(random_state=42)
scores = cross_validate(rf_model, X_train, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9712428402582052
평균 카파  : 0.9568398855320541


In [237]:
rf_model = RandomForestClassifier(random_state=42)

rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9687875150060024
[[291   1   0]
 [  0 242  12]
 [  0  13 274]]
              precision    recall  f1-score   support

      LAYING       1.00      1.00      1.00       292
     SITTING       0.95      0.95      0.95       254
    STANDING       0.96      0.95      0.96       287

    accuracy                           0.97       833
   macro avg       0.97      0.97      0.97       833
weighted avg       0.97      0.97      0.97       833

Cohen's Kappa Score: 0.9531


In [238]:
joblib.dump(rf_model, 'rf_model_default.pkl')

['rf_model_default.pkl']

In [239]:
new_row = pd.DataFrame([{'모델': 'RandomForest', 'default_정확도(교차검증)': np.mean(scores['test_accuracy']), 'default_정확도(테스트)': accuracy_score(y_test, y_pred)}])
acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [240]:
new_row = pd.DataFrame([{'모델': 'RandomForest', 'default_kappa(교차검증)': np.mean(scores['test_kappa']), 'default_kappa(테스트)': cohen_kappa_score(y_test, y_pred)}])
kappa_df = pd.concat([kappa_df, new_row], ignore_index=True)

In [241]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트)
0,RandomForest,0.971243,0.968788,NaN,NaN


In [242]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트)
0,RandomForest,0.95684,0.953102


### (2) 알고리즘2 : LogisticRegression

In [243]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
scores = cross_validate(lr_model, X_train, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9734081088711413
평균 카파  : 0.9600933526251557


In [244]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_test)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9675870348139256
[[292   0   0]
 [  0 240  14]
 [  0  13 274]]
              precision    recall  f1-score   support

      LAYING       1.00      1.00      1.00       292
     SITTING       0.95      0.94      0.95       254
    STANDING       0.95      0.95      0.95       287

    accuracy                           0.97       833
   macro avg       0.97      0.97      0.97       833
weighted avg       0.97      0.97      0.97       833

Cohen's Kappa Score: 0.9513


In [245]:
joblib.dump(lr_model, 'lr_model_default.pkl')

['lr_model_default.pkl']

In [246]:
new_row = pd.DataFrame([{'모델': 'LogisticRegression', 'default_정확도(교차검증)': np.mean(scores['test_accuracy']), 'default_정확도(테스트)': accuracy_score(y_test, y_pred)}])
acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [247]:
new_row = pd.DataFrame([{'모델': 'LogisticRegression', 'default_kappa(교차검증)': np.mean(scores['test_kappa']), 'default_kappa(테스트)': cohen_kappa_score(y_test, y_pred)}])
kappa_df = pd.concat([kappa_df, new_row], ignore_index=True)

In [248]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트)
0,RandomForest,0.971243,0.968788,NaN,NaN
1,LogisticRegression,0.973408,0.967587,NaN,NaN


In [249]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트)
0,RandomForest,0.956840,0.953102
1,LogisticRegression,0.960093,0.951287


### (3) 알고리즘3 : SVM

In [250]:
svm_model = SVC(random_state=42)
scores = cross_validate(svm_model, X_train, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9477430962623397
평균 카파  : 0.9215606349016531


In [251]:
svm_model = SVC(random_state=42)
svm_model.fit(X_train, y_train)

y_pred = svm_model.predict(X_test)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9531812725090036
[[292   0   0]
 [  0 230  24]
 [  0  15 272]]
              precision    recall  f1-score   support

      LAYING       1.00      1.00      1.00       292
     SITTING       0.94      0.91      0.92       254
    STANDING       0.92      0.95      0.93       287

    accuracy                           0.95       833
   macro avg       0.95      0.95      0.95       833
weighted avg       0.95      0.95      0.95       833

Cohen's Kappa Score: 0.9296


In [252]:
joblib.dump(svm_model, 'svm_model_default.pkl')

['svm_model_default.pkl']

In [253]:
new_row = pd.DataFrame([{'모델': 'SVM', 'default_정확도(교차검증)': np.mean(scores['test_accuracy']), 'default_정확도(테스트)': accuracy_score(y_test, y_pred)}])
acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [254]:
new_row = pd.DataFrame([{'모델': 'SVM', 'default_kappa(교차검증)': np.mean(scores['test_kappa']), 'default_kappa(테스트)': cohen_kappa_score(y_test, y_pred)}])
kappa_df = pd.concat([kappa_df, new_row], ignore_index=True)

In [255]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트)
0,RandomForest,0.971243,0.968788,NaN,NaN
1,LogisticRegression,0.973408,0.967587,NaN,NaN
2,SVM,0.947743,0.953181,NaN,NaN


In [256]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트)
0,RandomForest,0.956840,0.953102
1,LogisticRegression,0.960093,0.951287
2,SVM,0.921561,0.929597


### (4) 알고리즘4 : DecisionTreeClassifier

In [257]:
dt_model = DecisionTreeClassifier(random_state=42)
scores = cross_validate(dt_model, X_train, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9384695259377647
평균 카파  : 0.9076438801461137


In [258]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9375750300120048
[[290   2   0]
 [  0 234  20]
 [  0  30 257]]
              precision    recall  f1-score   support

      LAYING       1.00      0.99      1.00       292
     SITTING       0.88      0.92      0.90       254
    STANDING       0.93      0.90      0.91       287

    accuracy                           0.94       833
   macro avg       0.94      0.94      0.94       833
weighted avg       0.94      0.94      0.94       833

Cohen's Kappa Score: 0.9063


In [259]:
joblib.dump(dt_model, 'dt_model_default.pkl')

['dt_model_default.pkl']

In [260]:
new_row = pd.DataFrame([{'모델': 'DecisionTreeClassifier', 'default_정확도(교차검증)': np.mean(scores['test_accuracy']), 'default_정확도(테스트)': accuracy_score(y_test, y_pred)}])
acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [261]:
new_row = pd.DataFrame([{'모델': 'DecisionTreeClassifier', 'default_kappa(교차검증)': np.mean(scores['test_kappa']), 'default_kappa(테스트)': cohen_kappa_score(y_test, y_pred)}])
kappa_df = pd.concat([kappa_df, new_row], ignore_index=True)

In [262]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트)
0,RandomForest,0.971243,0.968788,NaN,NaN
1,LogisticRegression,0.973408,0.967587,NaN,NaN
2,SVM,0.947743,0.953181,NaN,NaN
3,DecisionTreeClassifier,0.938470,0.937575,NaN,NaN


In [263]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트)
0,RandomForest,0.956840,0.953102
1,LogisticRegression,0.960093,0.951287
2,SVM,0.921561,0.929597
3,DecisionTreeClassifier,0.907644,0.906272


In [264]:
acc_df.to_csv('모델 정확도(디폴트).csv', index=False, encoding='utf-8-sig')

In [265]:
kappa_df.to_csv('모델 kappa(디폴트).csv', index=False, encoding='utf-8-sig')

## ================일단 여기까지=====================

### (5) 알고리즘5 : XGBoost

In [266]:
# le = LabelEncoder()
# y_xgb = le.fit_transform(y)

In [267]:
# X_train_xgb, X_val_xgb, y_train_xgb, y_val_xgb = train_test_split(X, y_xgb, test_size=0.2, random_state=42)

In [268]:
# XGB_model = XGBClassifier(random_state=42)
# XGB_model.fit(X_train_xgb, y_train_xgb)

# y_pred = XGB_model.predict(X_val_xgb)
# y_pred = le.inverse_transform(y_pred)

# y_val = le.inverse_transform(y_val_xgb)

# print('accuracy :',accuracy_score(y_val, y_pred))
# print('='*60)
# print(confusion_matrix(y_val, y_pred))
# print('='*60)
# print(classification_report(y_val, y_pred))

In [269]:
# new_row = pd.DataFrame([{'model': 'XGBoost', 'accuracy': accuracy_score(y_val, y_pred)}])
# acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [270]:
# acc_df

### (6) 알고리즘6 : LightGBM

In [271]:
# X_lgb = X.copy()
# X_lgb.columns = [f'feature_{i}' for i in range(X.shape[1])]

In [272]:
# le = LabelEncoder()
# y_lgb = le.fit_transform(y)

In [273]:
# X_train_lgb, X_val_lgb, y_train_lgb, y_val_lgb = train_test_split(X_lgb, y_lgb, test_size=0.2, random_state=42)

In [274]:
# LGB_model = lgb.LGBMClassifier(random_state=42)
# LGB_model.fit(X_train_lgb, y_train_lgb)  # 경고문이 나옴. 너무 많은 특성 중 쓸모 없는 게 많을 때 이런 경고가 뜰 수 있음. 특성 중요도로 피처 수를 조절할 필요가 있음.

# y_pred = LGB_model.predict(X_val_lgb)
# y_pred = le.inverse_transform(y_pred)

# y_val= le.inverse_transform(y_val_lgb)

# print('accuracy :',accuracy_score(y_val, y_pred))
# print('='*60)
# print(confusion_matrix(y_val, y_pred))
# print('='*60)
# print(classification_report(y_val, y_pred))

### (7) 알고리즘7 : ANN

In [275]:
# X = data.drop('Activity', axis=1)
# y = data.loc[:, 'Activity'].values

In [276]:
# from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder()
# y_encoded = le.fit_transform(y)

In [277]:
# x_train, x_val, y_train, y_val = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

In [278]:
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Input, Dense, Dropout
# model = Sequential()

In [279]:
# model.add(Input(shape=(561,)))
# model.add(Dense(283, activation='relu'))
# model.add(Dropout(0.3))
# model.add(Dense(units=283, activation='relu'))
# model.add(Dropout(0.3))
# model.add(Dense(units=283, activation='relu'))
# model.add(Dropout(0.3))
# model.add(Dense(units=6, activation='softmax'))

In [280]:
# model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [281]:
# model.fit(x_train, y_train, batch_size = 32, epochs = 50)

In [282]:
# y_pred = model.predict(x_val)
# y_pred = np.argmax(y_pred, axis=1)

In [283]:
# y_val = le.inverse_transform(y_val)
# y_pred = le.inverse_transform(y_pred)
# print('accuracy :',accuracy_score(y_val, y_pred))
# print('='*60)
# print(confusion_matrix(y_val, y_pred))
# print('='*60)
# print(classification_report(y_val, y_pred))

## 특성 중요도(랜덤 포레스트에서 추출)

In [284]:
acc_df = pd.read_csv('모델 정확도(디폴트).csv')
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트)
0,RandomForest,0.971243,0.968788,NaN,NaN
1,LogisticRegression,0.973408,0.967587,NaN,NaN
2,SVM,0.947743,0.953181,NaN,NaN
3,DecisionTreeClassifier,0.938470,0.937575,NaN,NaN


In [285]:
rf_model = joblib.load('rf_model_default.pkl')

In [286]:
RFmodel_fi_df = plot_feature_importance(rf_model.feature_importances_, list(X_train), True)
RFmodel_fi_df

,feature_name,feature_importance
0,tGravityAcc-mean()-X,0.067013
1,"angle(X,gravityMean)",0.058918
2,tGravityAcc-mean()-Y,0.056230
3,tGravityAcc-max()-X,0.055631
4,"angle(Z,gravityMean)",0.045207
...,...,...
556,fBodyAccJerk-entropy()-Z,0.000066
557,fBodyBodyGyroJerkMag-entropy(),0.000044
558,fBodyBodyAccJerkMag-entropy(),0.000037
559,fBodyBodyGyroJerkMag-maxInds,0.000028


In [287]:
# 기존 변수 개수
len(RFmodel_fi_df)

561

In [288]:
RFmodel_fi_df['cum_importance'] = RFmodel_fi_df['feature_importance'].cumsum() / RFmodel_fi_df['feature_importance'].sum()

In [289]:
# 누적 중요도 기준 95%까지 필요한 변수 개수
selected_features = RFmodel_fi_df[RFmodel_fi_df['cum_importance'] <= 0.95]['feature_importance']
print(f"누적 중요도 95%까지 선택된 변수 개수: {len(selected_features)}")

누적 중요도 95%까지 선택된 변수 개수: 319


In [290]:
selected_cum_importance = RFmodel_fi_df.loc[:len(selected_features), 'feature_importance'].sum()
print(f"상위 {len(selected_features)+1}개 변수의 누적 중요도: {selected_cum_importance:.4f}")

상위 320개 변수의 누적 중요도: 0.9501


In [291]:
X_train_fi = train_data[RFmodel_fi_df['feature_name'][:len(selected_features)+1]]
y_train_fi = train_data.loc[:, 'Activity']

In [292]:
X_test_fi = test_data[RFmodel_fi_df['feature_name'][:len(selected_features)+1]]

### 랜덤 포레스트

In [293]:
rf_model = RandomForestClassifier(random_state=42)
scores = cross_validate(rf_model, X_train_fi, y_train_fi, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9724797948138825
평균 카파  : 0.9586965537547906


In [294]:
random_forest_model_fi = RandomForestClassifier(random_state=42)

random_forest_model_fi.fit(X_train_fi, y_train_fi)
y_pred = random_forest_model_fi.predict(X_test_fi)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9663865546218487
[[290   2   0]
 [  0 244  10]
 [  0  16 271]]
              precision    recall  f1-score   support

      LAYING       1.00      0.99      1.00       292
     SITTING       0.93      0.96      0.95       254
    STANDING       0.96      0.94      0.95       287

    accuracy                           0.97       833
   macro avg       0.97      0.97      0.97       833
weighted avg       0.97      0.97      0.97       833

Cohen's Kappa Score: 0.9495


In [295]:
joblib.dump(rf_model, 'rf_model_default_fi.pkl')

['rf_model_default_fi.pkl']

In [296]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트)
0,RandomForest,0.971243,0.968788,NaN,NaN
1,LogisticRegression,0.973408,0.967587,NaN,NaN
2,SVM,0.947743,0.953181,NaN,NaN
3,DecisionTreeClassifier,0.938470,0.937575,NaN,NaN


In [297]:
acc_df.loc[0, 'default_정확도(fi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[0, 'default_정확도(fi_테스트)'] = accuracy_score(y_test, y_pred)

In [298]:
kappa_df.loc[0, 'default_kappa(fi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[0, 'default_kappa(fi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [299]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트)
0,RandomForest,0.971243,0.968788,0.97248,0.966387
1,LogisticRegression,0.973408,0.967587,NaN,NaN
2,SVM,0.947743,0.953181,NaN,NaN
3,DecisionTreeClassifier,0.938470,0.937575,NaN,NaN


In [300]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트),default_kappa(fi_교차검증),default_kappa(fi_테스트)
0,RandomForest,0.956840,0.953102,0.958697,0.949517
1,LogisticRegression,0.960093,0.951287,NaN,NaN
2,SVM,0.921561,0.929597,NaN,NaN
3,DecisionTreeClassifier,0.907644,0.906272,NaN,NaN


### LinearRegression

In [301]:
# 랜덤 포레스트에서 추출한 중요도 상위 특성 활용
lr_model = LogisticRegression(max_iter=1000, random_state=42)
scores = cross_validate(lr_model, X_train_fi, y_train_fi, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9724836229130878
평균 카파  : 0.9587058375154698


In [302]:
logistic_model = LogisticRegression(max_iter=1000, random_state=42)
logistic_model.fit(X_train_fi, y_train_fi)
y_pred = logistic_model.predict(X_test_fi)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9747899159663865
[[292   0   0]
 [  0 243  11]
 [  0  10 277]]
              precision    recall  f1-score   support

      LAYING       1.00      1.00      1.00       292
     SITTING       0.96      0.96      0.96       254
    STANDING       0.96      0.97      0.96       287

    accuracy                           0.97       833
   macro avg       0.97      0.97      0.97       833
weighted avg       0.97      0.97      0.97       833

Cohen's Kappa Score: 0.9621


In [303]:
joblib.dump(lr_model, 'lr_model_default_fi.pkl')

['lr_model_default_fi.pkl']

In [304]:
acc_df.loc[1, 'default_정확도(fi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[1, 'default_정확도(fi_테스트)'] = accuracy_score(y_test, y_pred)

In [305]:
kappa_df.loc[1, 'default_kappa(fi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[1, 'default_kappa(fi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [306]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트)
0,RandomForest,0.971243,0.968788,0.972480,0.966387
1,LogisticRegression,0.973408,0.967587,0.972484,0.974790
2,SVM,0.947743,0.953181,NaN,NaN
3,DecisionTreeClassifier,0.938470,0.937575,NaN,NaN


In [307]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트),default_kappa(fi_교차검증),default_kappa(fi_테스트)
0,RandomForest,0.956840,0.953102,0.958697,0.949517
1,LogisticRegression,0.960093,0.951287,0.958706,0.962112
2,SVM,0.921561,0.929597,NaN,NaN
3,DecisionTreeClassifier,0.907644,0.906272,NaN,NaN


### SVM

In [308]:
svm_model = SVC(random_state=42)
scores = cross_validate(svm_model, X_train_fi, y_train_fi, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9502174838860948
평균 카파  : 0.9252789261274114


In [309]:
svm_model = SVC(random_state=42)
svm_model.fit(X_train_fi, y_train_fi)

y_pred = svm_model.predict(X_test_fi)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9603841536614646
[[292   0   0]
 [  0 235  19]
 [  0  14 273]]
              precision    recall  f1-score   support

      LAYING       1.00      1.00      1.00       292
     SITTING       0.94      0.93      0.93       254
    STANDING       0.93      0.95      0.94       287

    accuracy                           0.96       833
   macro avg       0.96      0.96      0.96       833
weighted avg       0.96      0.96      0.96       833

Cohen's Kappa Score: 0.9404


In [310]:
joblib.dump(svm_model, 'svm_model_default_fi.pkl')

['svm_model_default_fi.pkl']

In [311]:
acc_df.loc[2, 'default_정확도(fi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[2, 'default_정확도(fi_테스트)'] = accuracy_score(y_test, y_pred)

In [312]:
kappa_df.loc[2, 'default_kappa(fi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[2, 'default_kappa(fi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [313]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트)
0,RandomForest,0.971243,0.968788,0.972480,0.966387
1,LogisticRegression,0.973408,0.967587,0.972484,0.974790
2,SVM,0.947743,0.953181,0.950217,0.960384
3,DecisionTreeClassifier,0.938470,0.937575,NaN,NaN


In [314]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트),default_kappa(fi_교차검증),default_kappa(fi_테스트)
0,RandomForest,0.956840,0.953102,0.958697,0.949517
1,LogisticRegression,0.960093,0.951287,0.958706,0.962112
2,SVM,0.921561,0.929597,0.925279,0.940445
3,DecisionTreeClassifier,0.907644,0.906272,NaN,NaN


### DecisionTreeClassifier

In [315]:
dt_model = DecisionTreeClassifier(random_state=42)
scores = cross_validate(dt_model, X_train_fi, y_train_fi, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9434149515984707
평균 카파  : 0.915073393041945


In [316]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_fi, y_train_fi)

y_pred = dt_model.predict(X_test_fi)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9459783913565426
[[290   2   0]
 [  0 237  17]
 [  0  26 261]]
              precision    recall  f1-score   support

      LAYING       1.00      0.99      1.00       292
     SITTING       0.89      0.93      0.91       254
    STANDING       0.94      0.91      0.92       287

    accuracy                           0.95       833
   macro avg       0.94      0.95      0.94       833
weighted avg       0.95      0.95      0.95       833

Cohen's Kappa Score: 0.9189


In [317]:
joblib.dump(dt_model, 'dt_model_default_fi.pkl')

['dt_model_default_fi.pkl']

In [318]:
acc_df.loc[3, 'default_정확도(fi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[3, 'default_정확도(fi_테스트)'] = accuracy_score(y_test, y_pred)

In [319]:
kappa_df.loc[3, 'default_kappa(fi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[3, 'default_kappa(fi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [320]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트)
0,RandomForest,0.971243,0.968788,0.972480,0.966387
1,LogisticRegression,0.973408,0.967587,0.972484,0.974790
2,SVM,0.947743,0.953181,0.950217,0.960384
3,DecisionTreeClassifier,0.938470,0.937575,0.943415,0.945978


In [321]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트),default_kappa(fi_교차검증),default_kappa(fi_테스트)
0,RandomForest,0.956840,0.953102,0.958697,0.949517
1,LogisticRegression,0.960093,0.951287,0.958706,0.962112
2,SVM,0.921561,0.929597,0.925279,0.940445
3,DecisionTreeClassifier,0.907644,0.906272,0.915073,0.918883


In [322]:
acc_df.to_csv('모델 정확도(디폴트_fi).csv', index=False, encoding='utf-8-sig')

In [323]:
kappa_df.to_csv('모델 kappa(디폴트_fi).csv', index=False, encoding='utf-8-sig')

##방법2. PCA 활용

In [324]:
acc_df = pd.read_csv('모델 정확도(디폴트_fi).csv')
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트)
0,RandomForest,0.971243,0.968788,0.972480,0.966387
1,LogisticRegression,0.973408,0.967587,0.972484,0.974790
2,SVM,0.947743,0.953181,0.950217,0.960384
3,DecisionTreeClassifier,0.938470,0.937575,0.943415,0.945978


In [325]:
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train)

print("선택된 주성분 수:", pca.n_components_)
print("누적 설명력:", pca.explained_variance_ratio_.cumsum()[-1])

선택된 주성분 수: 76
누적 설명력: 0.9508246619476548


In [326]:
X_test_pca = pca.transform(X_test)

### 랜덤포레스트

In [327]:
rf_model = RandomForestClassifier(random_state=42)
scores = cross_validate(rf_model, X_train_pca, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9236217646580311
평균 카파  : 0.8853349335593608


In [328]:
rf_model = RandomForestClassifier(random_state=42)

rf_model.fit(X_train_pca, y_train)
y_pred = rf_model.predict(X_test_pca)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9243697478991597
[[291   1   0]
 [  2 209  43]
 [  0  17 270]]
              precision    recall  f1-score   support

      LAYING       0.99      1.00      0.99       292
     SITTING       0.92      0.82      0.87       254
    STANDING       0.86      0.94      0.90       287

    accuracy                           0.92       833
   macro avg       0.93      0.92      0.92       833
weighted avg       0.93      0.92      0.92       833

Cohen's Kappa Score: 0.8861


In [329]:
acc_df.loc[0, 'default_정확도(pca_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[0, 'default_정확도(pca_테스트)'] = accuracy_score(y_test, y_pred)

In [330]:
kappa_df.loc[0, 'default_kappa(pca_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[0, 'default_kappa(pca_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [331]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트),default_정확도(pca_교차검증),default_정확도(pca_테스트)
0,RandomForest,0.971243,0.968788,0.972480,0.966387,0.923622,0.92437
1,LogisticRegression,0.973408,0.967587,0.972484,0.974790,NaN,NaN
2,SVM,0.947743,0.953181,0.950217,0.960384,NaN,NaN
3,DecisionTreeClassifier,0.938470,0.937575,0.943415,0.945978,NaN,NaN


In [332]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트),default_kappa(fi_교차검증),default_kappa(fi_테스트),default_kappa(pca_교차검증),default_kappa(pca_테스트)
0,RandomForest,0.956840,0.953102,0.958697,0.949517,0.885335,0.886124
1,LogisticRegression,0.960093,0.951287,0.958706,0.962112,NaN,NaN
2,SVM,0.921561,0.929597,0.925279,0.940445,NaN,NaN
3,DecisionTreeClassifier,0.907644,0.906272,0.915073,0.918883,NaN,NaN


### 로지스틱회귀

In [333]:
lr_model = LogisticRegression(max_iter=1000, random_state=42) # 최적점에 도달 못하여 max_iter=1000로 설정
scores = cross_validate(lr_model, X_train_pca, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9610390418267689
평균 카파  : 0.9415234640722545


In [334]:
lr_model = LogisticRegression(max_iter=1000, random_state=42) # 최적점에 도달 못하여 max_iter=1000로 설정
lr_model.fit(X_train_pca, y_train)
y_pred = lr_model.predict(X_test_pca)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9555822328931572
[[292   0   0]
 [  0 237  17]
 [  0  20 267]]
              precision    recall  f1-score   support

      LAYING       1.00      1.00      1.00       292
     SITTING       0.92      0.93      0.93       254
    STANDING       0.94      0.93      0.94       287

    accuracy                           0.96       833
   macro avg       0.95      0.95      0.95       833
weighted avg       0.96      0.96      0.96       833

Cohen's Kappa Score: 0.9333


In [335]:
acc_df.loc[1, 'default_정확도(pca_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[1, 'default_정확도(pca_테스트)'] = accuracy_score(y_test, y_pred)

In [336]:
kappa_df.loc[1, 'default_kappa(pca_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[1, 'default_kappa(pca_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [337]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트),default_정확도(pca_교차검증),default_정확도(pca_테스트)
0,RandomForest,0.971243,0.968788,0.972480,0.966387,0.923622,0.924370
1,LogisticRegression,0.973408,0.967587,0.972484,0.974790,0.961039,0.955582
2,SVM,0.947743,0.953181,0.950217,0.960384,NaN,NaN
3,DecisionTreeClassifier,0.938470,0.937575,0.943415,0.945978,NaN,NaN


In [338]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트),default_kappa(fi_교차검증),default_kappa(fi_테스트),default_kappa(pca_교차검증),default_kappa(pca_테스트)
0,RandomForest,0.956840,0.953102,0.958697,0.949517,0.885335,0.886124
1,LogisticRegression,0.960093,0.951287,0.958706,0.962112,0.941523,0.933265
2,SVM,0.921561,0.929597,0.925279,0.940445,NaN,NaN
3,DecisionTreeClassifier,0.907644,0.906272,0.915073,0.918883,NaN,NaN


### SVM

In [339]:
svm_model = SVC(random_state=42)
scores = cross_validate(svm_model, X_train_pca, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9672238146051555
평균 카파  : 0.9508109405833753


In [340]:
svm_model = SVC(random_state=42)
svm_model.fit(X_train_pca, y_train)

y_pred = svm_model.predict(X_test_pca)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9723889555822329
[[292   0   0]
 [  0 246   8]
 [  0  15 272]]
              precision    recall  f1-score   support

      LAYING       1.00      1.00      1.00       292
     SITTING       0.94      0.97      0.96       254
    STANDING       0.97      0.95      0.96       287

    accuracy                           0.97       833
   macro avg       0.97      0.97      0.97       833
weighted avg       0.97      0.97      0.97       833

Cohen's Kappa Score: 0.9585


In [341]:
acc_df.loc[2, 'default_정확도(pca_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[2, 'default_정확도(pca_테스트)'] = accuracy_score(y_test, y_pred)

In [342]:
kappa_df.loc[2, 'default_kappa(pca_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[2, 'default_kappa(pca_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [343]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트),default_정확도(pca_교차검증),default_정확도(pca_테스트)
0,RandomForest,0.971243,0.968788,0.972480,0.966387,0.923622,0.924370
1,LogisticRegression,0.973408,0.967587,0.972484,0.974790,0.961039,0.955582
2,SVM,0.947743,0.953181,0.950217,0.960384,0.967224,0.972389
3,DecisionTreeClassifier,0.938470,0.937575,0.943415,0.945978,NaN,NaN


In [344]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트),default_kappa(fi_교차검증),default_kappa(fi_테스트),default_kappa(pca_교차검증),default_kappa(pca_테스트)
0,RandomForest,0.956840,0.953102,0.958697,0.949517,0.885335,0.886124
1,LogisticRegression,0.960093,0.951287,0.958706,0.962112,0.941523,0.933265
2,SVM,0.921561,0.929597,0.925279,0.940445,0.950811,0.958528
3,DecisionTreeClassifier,0.907644,0.906272,0.915073,0.918883,NaN,NaN


### 결정 트리

In [345]:
dt_model = DecisionTreeClassifier(random_state=42)
scores = cross_validate(dt_model, X_train_pca, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.8320971762983238
평균 카파  : 0.7479755974734039


In [346]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_pca, y_train)

y_pred = dt_model.predict(X_test_pca)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.8211284513805522
[[282   8   2]
 [  3 184  67]
 [  0  69 218]]
              precision    recall  f1-score   support

      LAYING       0.99      0.97      0.98       292
     SITTING       0.70      0.72      0.71       254
    STANDING       0.76      0.76      0.76       287

    accuracy                           0.82       833
   macro avg       0.82      0.82      0.82       833
weighted avg       0.82      0.82      0.82       833

Cohen's Kappa Score: 0.7314


In [347]:
acc_df.loc[3, 'default_정확도(pca_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[3, 'default_정확도(pca_테스트)'] = accuracy_score(y_test, y_pred)

In [348]:
kappa_df.loc[3, 'default_kappa(pca_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[3, 'default_kappa(pca_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [349]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트),default_정확도(pca_교차검증),default_정확도(pca_테스트)
0,RandomForest,0.971243,0.968788,0.972480,0.966387,0.923622,0.924370
1,LogisticRegression,0.973408,0.967587,0.972484,0.974790,0.961039,0.955582
2,SVM,0.947743,0.953181,0.950217,0.960384,0.967224,0.972389
3,DecisionTreeClassifier,0.938470,0.937575,0.943415,0.945978,0.832097,0.821128


In [350]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트),default_kappa(fi_교차검증),default_kappa(fi_테스트),default_kappa(pca_교차검증),default_kappa(pca_테스트)
0,RandomForest,0.956840,0.953102,0.958697,0.949517,0.885335,0.886124
1,LogisticRegression,0.960093,0.951287,0.958706,0.962112,0.941523,0.933265
2,SVM,0.921561,0.929597,0.925279,0.940445,0.950811,0.958528
3,DecisionTreeClassifier,0.907644,0.906272,0.915073,0.918883,0.747976,0.731352


In [351]:
acc_df.to_csv('모델 정확도(디폴트_fi_pca).csv', index=False, encoding='utf-8-sig')

In [352]:
kappa_df.to_csv('모델 kappa(디폴트_fi_pca).csv', index=False, encoding='utf-8-sig')

##방법3. permutation_importance()

In [353]:
acc_df = pd.read_csv('모델 정확도(디폴트_fi_pca).csv')
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트),default_정확도(pca_교차검증),default_정확도(pca_테스트)
0,RandomForest,0.971243,0.968788,0.972480,0.966387,0.923622,0.924370
1,LogisticRegression,0.973408,0.967587,0.972484,0.974790,0.961039,0.955582
2,SVM,0.947743,0.953181,0.950217,0.960384,0.967224,0.972389
3,DecisionTreeClassifier,0.938470,0.937575,0.943415,0.945978,0.832097,0.821128


In [354]:
rf_model = joblib.load('rf_model_default.pkl')
lr_model = joblib.load('lr_model_default.pkl')
svm_model = joblib.load('svm_model_default.pkl')
dt_model = joblib.load('dt_model_default.pkl')

### 랜덤포레스트

In [355]:
model = rf_model

# permutation importance 계산
result_rf = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42, n_jobs=-1)

In [356]:
# 중요도 정리 + 누적 중요도 계산
importances_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': result_rf.importances_mean
}).sort_values(by='importance', ascending=False)

# 중요도가 양수인 피처만 선택
importances_df = importances_df[importances_df['importance'] > 0]

print(f"선택된 피처 수: {len(importances_df)}")

# 피처 이름 추출
selected_features = importances_df['feature']

# 선택된 피처로 데이터 재구성
X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

선택된 피처 수: 165


In [357]:
importances_df.to_csv('rf_permutation_importance_DataFrame(디폴트).csv', index=False, encoding='utf-8-sig')

In [358]:
rf_model = RandomForestClassifier(random_state=42)
scores = cross_validate(rf_model, X_train_sel, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9752623444236557
평균 카파  : 0.9628757998599842


In [359]:
rf_model = RandomForestClassifier(random_state=42)

rf_model.fit(X_train_sel, y_train)
y_pred = rf_model.predict(X_test_sel)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.978391356542617
[[292   0   0]
 [  0 242  12]
 [  0   6 281]]
              precision    recall  f1-score   support

      LAYING       1.00      1.00      1.00       292
     SITTING       0.98      0.95      0.96       254
    STANDING       0.96      0.98      0.97       287

    accuracy                           0.98       833
   macro avg       0.98      0.98      0.98       833
weighted avg       0.98      0.98      0.98       833

Cohen's Kappa Score: 0.9675


In [360]:
acc_df.loc[0, 'default_정확도(pi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[0, 'default_정확도(pi_테스트)'] = accuracy_score(y_test, y_pred)

In [361]:
kappa_df.loc[0, 'default_kappa(pi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[0, 'default_kappa(pi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [362]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트),default_정확도(pca_교차검증),default_정확도(pca_테스트),default_정확도(pi_교차검증),default_정확도(pi_테스트)
0,RandomForest,0.971243,0.968788,0.972480,0.966387,0.923622,0.924370,0.975262,0.978391
1,LogisticRegression,0.973408,0.967587,0.972484,0.974790,0.961039,0.955582,NaN,NaN
2,SVM,0.947743,0.953181,0.950217,0.960384,0.967224,0.972389,NaN,NaN
3,DecisionTreeClassifier,0.938470,0.937575,0.943415,0.945978,0.832097,0.821128,NaN,NaN


In [363]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트),default_kappa(fi_교차검증),default_kappa(fi_테스트),default_kappa(pca_교차검증),default_kappa(pca_테스트),default_kappa(pi_교차검증),default_kappa(pi_테스트)
0,RandomForest,0.956840,0.953102,0.958697,0.949517,0.885335,0.886124,0.962876,0.967513
1,LogisticRegression,0.960093,0.951287,0.958706,0.962112,0.941523,0.933265,NaN,NaN
2,SVM,0.921561,0.929597,0.925279,0.940445,0.950811,0.958528,NaN,NaN
3,DecisionTreeClassifier,0.907644,0.906272,0.915073,0.918883,0.747976,0.731352,NaN,NaN


### 로지스틱회귀

In [364]:
model = lr_model

# permutation importance 계산
result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42, n_jobs=-1)

In [365]:
# 중요도 정리 + 누적 중요도 계산
importances_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': result.importances_mean
}).sort_values(by='importance', ascending=False)

# 중요도가 양수인 피처만 선택
importances_df = importances_df[importances_df['importance'] > 0]

print(f"선택된 피처 수: {len(importances_df)}")

# 피처 이름 추출
selected_features = importances_df['feature']

# 선택된 피처로 데이터 재구성
X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

선택된 피처 수: 101


In [366]:
importances_df.to_csv('lr_permutation_importance_DataFrame(디폴트).csv', index=False, encoding='utf-8-sig')

In [367]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
scores = cross_validate(lr_model, X_train_sel, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9659878170742795
평균 카파  : 0.9489557292793063


In [368]:
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_sel, y_train)
y_pred = lr_model.predict(X_test_sel)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9591836734693877
[[290   2   0]
 [  1 240  13]
 [  0  18 269]]
              precision    recall  f1-score   support

      LAYING       1.00      0.99      0.99       292
     SITTING       0.92      0.94      0.93       254
    STANDING       0.95      0.94      0.95       287

    accuracy                           0.96       833
   macro avg       0.96      0.96      0.96       833
weighted avg       0.96      0.96      0.96       833

Cohen's Kappa Score: 0.9387


In [369]:
acc_df.loc[1, 'default_정확도(pi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[1, 'default_정확도(pi_테스트)'] = accuracy_score(y_test, y_pred)

In [370]:
kappa_df.loc[1, 'default_kappa(pi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[1, 'default_kappa(pi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [371]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트),default_정확도(pca_교차검증),default_정확도(pca_테스트),default_정확도(pi_교차검증),default_정확도(pi_테스트)
0,RandomForest,0.971243,0.968788,0.972480,0.966387,0.923622,0.924370,0.975262,0.978391
1,LogisticRegression,0.973408,0.967587,0.972484,0.974790,0.961039,0.955582,0.965988,0.959184
2,SVM,0.947743,0.953181,0.950217,0.960384,0.967224,0.972389,NaN,NaN
3,DecisionTreeClassifier,0.938470,0.937575,0.943415,0.945978,0.832097,0.821128,NaN,NaN


In [372]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트),default_kappa(fi_교차검증),default_kappa(fi_테스트),default_kappa(pca_교차검증),default_kappa(pca_테스트),default_kappa(pi_교차검증),default_kappa(pi_테스트)
0,RandomForest,0.956840,0.953102,0.958697,0.949517,0.885335,0.886124,0.962876,0.967513
1,LogisticRegression,0.960093,0.951287,0.958706,0.962112,0.941523,0.933265,0.948956,0.938689
2,SVM,0.921561,0.929597,0.925279,0.940445,0.950811,0.958528,NaN,NaN
3,DecisionTreeClassifier,0.907644,0.906272,0.915073,0.918883,0.747976,0.731352,NaN,NaN


### SVM

In [373]:
model = svm_model

# permutation importance 계산
result = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1) # n_repeats=30 -> 너무 오래 걸림...30분이 넘게 걸려 결국 포기.  n_repeats=10 으로 실행
# 너무 오래 걸려서 안되면.... permutation_importance()를 랜덤포레스트로 하고 훈련을 svm으로 시키는 것을 고려할 것임.

In [374]:
# 중요도 정리 + 누적 중요도 계산
importances_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': result.importances_mean
}).sort_values(by='importance', ascending=False)

# 중요도가 양수인 피처만 선택
importances_df = importances_df[importances_df['importance'] > 0]

print(f"선택된 피처 수: {len(importances_df)}")

# 피처 이름 추출
selected_features = importances_df['feature']

# 선택된 피처로 데이터 재구성
X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

선택된 피처 수: 73


In [375]:
importances_df.to_csv('svm_permutation_importance_DataFrame(디폴트).csv', index=False, encoding='utf-8-sig')

In [376]:
svm_model = SVC(random_state=42)
scores = cross_validate(svm_model, X_train_sel, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.969388604705691
평균 카파  : 0.9540665388554779


In [377]:
svm_model = SVC(random_state=42)
svm_model.fit(X_train_sel, y_train)

y_pred = svm_model.predict(X_test_sel)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9831932773109243
[[292   0   0]
 [  0 247   7]
 [  0   7 280]]
              precision    recall  f1-score   support

      LAYING       1.00      1.00      1.00       292
     SITTING       0.97      0.97      0.97       254
    STANDING       0.98      0.98      0.98       287

    accuracy                           0.98       833
   macro avg       0.98      0.98      0.98       833
weighted avg       0.98      0.98      0.98       833

Cohen's Kappa Score: 0.9747


In [378]:
acc_df.loc[2, 'default_정확도(pi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[2, 'default_정확도(pi_테스트)'] = accuracy_score(y_test, y_pred)

In [379]:
kappa_df.loc[2, 'default_kappa(pi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[2, 'default_kappa(pi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [380]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트),default_정확도(pca_교차검증),default_정확도(pca_테스트),default_정확도(pi_교차검증),default_정확도(pi_테스트)
0,RandomForest,0.971243,0.968788,0.972480,0.966387,0.923622,0.924370,0.975262,0.978391
1,LogisticRegression,0.973408,0.967587,0.972484,0.974790,0.961039,0.955582,0.965988,0.959184
2,SVM,0.947743,0.953181,0.950217,0.960384,0.967224,0.972389,0.969389,0.983193
3,DecisionTreeClassifier,0.938470,0.937575,0.943415,0.945978,0.832097,0.821128,NaN,NaN


In [381]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트),default_kappa(fi_교차검증),default_kappa(fi_테스트),default_kappa(pca_교차검증),default_kappa(pca_테스트),default_kappa(pi_교차검증),default_kappa(pi_테스트)
0,RandomForest,0.956840,0.953102,0.958697,0.949517,0.885335,0.886124,0.962876,0.967513
1,LogisticRegression,0.960093,0.951287,0.958706,0.962112,0.941523,0.933265,0.948956,0.938689
2,SVM,0.921561,0.929597,0.925279,0.940445,0.950811,0.958528,0.954067,0.974743
3,DecisionTreeClassifier,0.907644,0.906272,0.915073,0.918883,0.747976,0.731352,NaN,NaN


### 결정트리

In [382]:
model = dt_model

# permutation importance 계산
result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42, n_jobs=-1)

In [383]:
# 중요도 정리 + 누적 중요도 계산
importances_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': result.importances_mean
}).sort_values(by='importance', ascending=False)

# 중요도가 양수인 피처만 선택
importances_df = importances_df[importances_df['importance'] > 0]

print(f"선택된 피처 수: {len(importances_df)}")

# 피처 이름 추출
selected_features = importances_df['feature']

# 선택된 피처로 데이터 재구성
X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

선택된 피처 수: 40


In [384]:
importances_df.to_csv('dt_permutation_importance_DataFrame(디폴트).csv', index=False, encoding='utf-8-sig')

In [385]:
dt_model = DecisionTreeClassifier(random_state=42)
scores = cross_validate(dt_model, X_train_sel, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9495982888396552
평균 카파  : 0.9243581987761725


In [386]:
dt_model = DecisionTreeClassifier( random_state=42)
dt_model.fit(X_train_sel, y_train)

y_pred = dt_model.predict(X_test_sel)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9483793517406963
[[290   2   0]
 [  0 234  20]
 [  0  21 266]]
              precision    recall  f1-score   support

      LAYING       1.00      0.99      1.00       292
     SITTING       0.91      0.92      0.92       254
    STANDING       0.93      0.93      0.93       287

    accuracy                           0.95       833
   macro avg       0.95      0.95      0.95       833
weighted avg       0.95      0.95      0.95       833

Cohen's Kappa Score: 0.9224


In [387]:
acc_df.loc[3, 'default_정확도(pi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[3, 'default_정확도(pi_테스트)'] = accuracy_score(y_test, y_pred)

In [388]:
kappa_df.loc[3, 'default_kappa(pi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[3, 'default_kappa(pi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [389]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트),default_정확도(pca_교차검증),default_정확도(pca_테스트),default_정확도(pi_교차검증),default_정확도(pi_테스트)
0,RandomForest,0.971243,0.968788,0.972480,0.966387,0.923622,0.924370,0.975262,0.978391
1,LogisticRegression,0.973408,0.967587,0.972484,0.974790,0.961039,0.955582,0.965988,0.959184
2,SVM,0.947743,0.953181,0.950217,0.960384,0.967224,0.972389,0.969389,0.983193
3,DecisionTreeClassifier,0.938470,0.937575,0.943415,0.945978,0.832097,0.821128,0.949598,0.948379


In [390]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트),default_kappa(fi_교차검증),default_kappa(fi_테스트),default_kappa(pca_교차검증),default_kappa(pca_테스트),default_kappa(pi_교차검증),default_kappa(pi_테스트)
0,RandomForest,0.956840,0.953102,0.958697,0.949517,0.885335,0.886124,0.962876,0.967513
1,LogisticRegression,0.960093,0.951287,0.958706,0.962112,0.941523,0.933265,0.948956,0.938689
2,SVM,0.921561,0.929597,0.925279,0.940445,0.950811,0.958528,0.954067,0.974743
3,DecisionTreeClassifier,0.907644,0.906272,0.915073,0.918883,0.747976,0.731352,0.924358,0.922444


In [391]:
acc_df.to_csv('모델 정확도(디폴트_fi_pca_pi).csv', index=False, encoding='utf-8-sig')

In [392]:
kappa_df.to_csv('모델 kappa(디폴트_fi_pca_pi).csv', index=False, encoding='utf-8-sig')